# 04 — Fairness Audit & Bias Mitigation (Home Credit)

Audits the tuned model's predictions for disparate impact across
`CODE_GENDER` and `AGE_GROUP` (Fairlearn: demographic parity ratio/
difference, equalized-odds difference, four-fifths-rule convention), then
applies reweighing (Kamiran & Calders, 2012) and re-audits — a direct
before/after comparison of both fairness metrics and predictive performance.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import joblib
from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_home_credit
from dac.features.engineering import engineer_home_credit_features, split_feature_columns
from dac.fairness.audit import audit_fairness
from dac.fairness.mitigation import compute_reweighing_weights
from dac.models.evaluate import compute_metrics

hc_cfg = CONFIG["data"]["home_credit"]
protected_attributes = hc_cfg["protected_attributes"]
df, _ = load_home_credit()
df = engineer_home_credit_features(df)
exclude_cols = [hc_cfg["id_col"], *protected_attributes]
numeric_cols, categorical_cols = split_feature_columns(df, hc_cfg["target_col"], exclude_cols)
X = df[numeric_cols + categorical_cols]
y = df[hc_cfg["target_col"]]
sensitive = df[protected_attributes]
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive, test_size=0.2, random_state=CONFIG["seed"], stratify=y
)

pipeline = joblib.load(CONFIG["paths"]["models_dir"] / "xgboost_tuned.joblib")
y_pred = (pipeline.predict_proba(X_test)[:, 1] >= 0.5).astype(int)

## Pre-mitigation audit

In [ ]:
pre = {}
for attr in protected_attributes:
    pre[attr] = audit_fairness(
        y_true=y_test.to_numpy(), y_pred=y_pred, sensitive_features=sens_test[attr],
        model_name="xgboost_tuned_pre_mitigation", attribute_name=attr,
        figures_dir=CONFIG["paths"]["figures_dir"] / "home_credit" / "fairness",
        metrics_dir=CONFIG["paths"]["metrics_dir"] / "fairness",
        favorable_label=CONFIG["fairness"]["favorable_label"],
    )
pre

## Reweighing mitigation + re-audit

In [ ]:
from dac.features.engineering import build_preprocessor
from dac.models.train import build_xgboost_pipeline, compute_scale_pos_weight

mitigation_attr = protected_attributes[0]
weights = compute_reweighing_weights(y_train, sens_train[mitigation_attr])

preprocessor = build_preprocessor(numeric_cols, categorical_cols)
mitigated_pipeline = build_xgboost_pipeline(preprocessor, scale_pos_weight=compute_scale_pos_weight(y_train))
mitigated_pipeline.fit(X_train, y_train, clf__sample_weight=weights)

proba_mitigated = mitigated_pipeline.predict_proba(X_test)[:, 1]
y_pred_mitigated = (proba_mitigated >= 0.5).astype(int)

print("Performance before:", compute_metrics(y_test.to_numpy(), pipeline.predict_proba(X_test)[:, 1]))
print("Performance after: ", compute_metrics(y_test.to_numpy(), proba_mitigated))

In [ ]:
post = {}
for attr in protected_attributes:
    post[attr] = audit_fairness(
        y_true=y_test.to_numpy(), y_pred=y_pred_mitigated, sensitive_features=sens_test[attr],
        model_name="xgboost_tuned_post_mitigation", attribute_name=attr,
        figures_dir=CONFIG["paths"]["figures_dir"] / "home_credit" / "fairness",
        metrics_dir=CONFIG["paths"]["metrics_dir"] / "fairness",
        favorable_label=CONFIG["fairness"]["favorable_label"],
    )
print(f"{mitigation_attr} demographic parity ratio: {pre[mitigation_attr]['demographic_parity_ratio']:.3f} -> {post[mitigation_attr]['demographic_parity_ratio']:.3f}")
post